# Examine Interpolation 

In [ ]:
from spyral.core.cluster import Cluster
from spyral.interpolate.track_interpolator import create_interpolator

# Pick one of these import lines to uncomment to use as your solver
# By default we chose the L-BFGS-B
from spyral.solvers.solver_interp import fit_model_interp, Guess, interpolate_trajectory
# from spyral.solvers.solver_interp_leastsq import fit_model_interp, Guess, interpolate_trajectory
from spyral.phases.interp_solver_phase import DEFAULT_PID_XAXIS, DEFAULT_PID_YAXIS

from spyral.core.run_stacks import form_run_string
from spyral import SolverParameters, DetectorParameters, InterpSolverPhase

from spyral_utils.nuclear import NuclearDataMap
from spyral_utils.nuclear.particle_id import deserialize_particle_id

import polars as pl
import h5py as h5
from pathlib import Path
import matplotlib.pyplot as plt

%matplotlib widget

In [ ]:
# Set some parameters
# workspace_path = Path("/path/to/your/workspace/")
# workspace_path = Path("/mnt/analysis/argonne2023/ramirezc/a2066/workspace_369/")

from json import load
from spyral_utils.nuclear import NuclearDataMap
nuclear_map = NuclearDataMap()
# Read the json file 
exp_tag = "a2066"

path_to_config = Path(f"../../notebooks/input_parameters/{exp_tag}.json")
with open(path_to_config,"r") as json_file:
    json_data = load(json_file)
    # Get keys from json and use them as their variables 
    workspace_path = Path(json_data["workspace_path"])
    trace_path = Path(json_data["trace_path"])
    target_path = Path(json_data["target_path"])
    run_min = json_data["run_min"]
    run_max = json_data["run_max"]
    runs_to_skip = json_data["runs_to_skip"]
    ejectile = nuclear_map.get_data(json_data["ejectile"][0], json_data["ejectile"][1])
    projectile = nuclear_map.get_data(json_data["projectile"][0], json_data["projectile"][1])
    target = nuclear_map.get_data(json_data["target"][0], json_data["target"][1])
    ic_pressure = json_data["ic_pressure"]
    beam_energy = json_data["beam_energy"]

workspace_path_pid = Path("/mnt/analysis/argonne2023/ramirezc/a2066/workspace_369_tune_pad")

solver_params = SolverParameters(
    gas_data_path=Path("/projects/attpc/ramirezc/a2066/gasfile.json"),
    particle_id_filename=workspace_path_pid/"hydrogen_1.json",
    # particle_id_filename = workspace_path/ "hydrogen_2.json",
    # #Second peak IC (7Be)
    ic_min_val = 420.0, #420(gate gaussian), 460 (extreme gate)
    ic_max_val = 600.0,

    #First peak IC (not that rigorous -7Li/ rigorus )
    # ic_min_val = 340.0,
    # ic_max_val = 400.0,#420.0,

    #SMALLEST PEAK BEFORE 7Li
    # ic_min_val = 260.0,
    # ic_max_val = 340.0,

    # red box
    # ic_min_val = 700.0,
    # ic_max_val = 1000.0,

    n_time_steps=1000,
    interp_ke_min=0.1,
    interp_ke_max=70.0,
    interp_ke_bins=350,
    interp_polar_min=5.0,
    #deuteron and he3 and t
    # interp_polar_max=90.0,
    # interp_polar_bins=170,
    #proton
    interp_polar_max=175.0,
    interp_polar_bins=340,#340,
    #
    fit_vertex_rho=True,
    fit_vertex_phi=True,
    fit_azimuthal=True,
    fit_method="lbfgsb",
    # fit_method ="emcee",
    # kwargs_fit_method={"steps": 1000,
    #                    "nwalkers": 24}
)

det_params = DetectorParameters(
    magnetic_field=2.85,
    electric_field=40000.0,
    detector_length=1000.0,
    beam_region_radius=25.0,
    #Magda Time Bucket
    # micromegas_time_bucket=10.0,
    # window_time_bucket=344.0,
    #Daniela Feb 2025
    micromegas_time_bucket= 20, # (other number = 12) 20, 368
    window_time_bucket= 369,#348.3,
    get_frequency=3.125,
    garfield_file_path=Path("E:/a2066/electrons_a2066.txt"),
    do_garfield_correction=False,
)

cluster_path = workspace_path / "Cluster" # this may change if you add custom phases
estimate_path = workspace_path / "Estimation" # this may change if you add custom phases

In [ ]:
nuc_map = NuclearDataMap()
pid = deserialize_particle_id(solver_params.particle_id_filename, nuc_map)
if pid is None:
    raise Exception("Particle ID error!")
pid_xaxis = DEFAULT_PID_XAXIS
pid_yaxis = DEFAULT_PID_YAXIS
if not pid.cut.is_default_x_axis() and not pid.cut.is_default_y_axis():
    pid_xaxis = pid.cut.get_x_axis()
    pid_yaxis = pid.cut.get_y_axis()
solver = InterpSolverPhase(solver_params, det_params)
success = solver.create_assets(workspace_path)
if not success:
    raise Exception("Could not setup interpolation mesh!")
tracks = create_interpolator(solver.track_path)

In [ ]:
run_number = 48
cluster_file_path = cluster_path / f"{form_run_string(run_number)}.h5"
cluster_file = h5.File(cluster_file_path, "r")
estimate_file_path = estimate_path / f"{form_run_string(run_number)}.parquet"
estimate_df = pl.scan_parquet(estimate_file_path)
import numpy as np
estimate_df = estimate_df.collect()
estimate_gated = estimate_df
# estimate_gated = estimate_df.filter((pl.col('polar')> np.pi/2-0.07) & (pl.col('polar')< np.pi/2 + 0.07) & (pl.col("ic_centroid")!=-1))
# estimate_gated = estimate_df.filter(pl.struct([pid_xaxis, pid_yaxis]).map_batches(pid.cut.is_cols_inside)).collect().to_dict()
cluster_group = cluster_file["cluster"]
nrows = len(estimate_df["event"])
row_iter = iter(range(nrows))
print(f"Number of rows: {nrows}")

In [ ]:
plt.close()
row = None
# Can always override with a hardcoded row if needed
# row = 16771
# row = 2409
if row is None:
    try:
        row = next(row_iter)
    except StopIteration:
        raise Exception("You ran out of rows for this run! Open a new run")
# find_event = 2673 #21307
find_event = 10093
row  = estimate_gated["event"] == find_event 
row = row.to_list().index(True) if True in row.to_list() else -1   
# row+=1
print(f"Row: {row}")
event = estimate_gated['event'][row]
cluster_index = estimate_gated['cluster_index'][row]
print(f"Event: {event}")
print(f"Cluster index: {cluster_index}")
event_group = cluster_group[f"event_{event}"]
local_cluster = event_group[f"cluster_{cluster_index}"]
print(f'Direction: {estimate_gated["direction"][row]}')
print(f"Vertex: ({estimate_gated["vertex_x"][row]:0.2f},{estimate_gated["vertex_y"][row]:0.2f},{estimate_gated["vertex_z"][row]:0.2f})")
cluster = Cluster(event, local_cluster.attrs["label"], local_cluster.attrs["direction"] , local_cluster["cloud"][:].copy())
# Forward 0, Backward 1

In [ ]:
from spyral.core.constants import QBRHO_2_P
from spyral.geometry.circle import generate_circle_points
import numpy as np
nuclear_map = NuclearDataMap()

guess = Guess(
    estimate_gated["polar"][row],
    estimate_gated["azimuthal"][row],
    estimate_gated["brho"][row],
    estimate_gated["vertex_x"][row],
    estimate_gated["vertex_y"][row],
    estimate_gated["vertex_z"][row]
)
print(guess)
result = fit_model_interp(cluster, guess, pid.nucleus, tracks, det_params, solver_params)
if result is None:
    print("Guess outside of interpolation range!")

best_fit_trajectory = interpolate_trajectory(result, tracks, pid.nucleus)

print(f"Vertex: ({result["vertex_x"].value*1000:0.2f}, {result["vertex_y"].value*1000:0.2f}, {result["vertex_z"].value*1000:0.2f})")
ejectile = nuclear_map.get_data(1,1)
ke = np.sqrt((result["brho"].value*float(ejectile.Z)*QBRHO_2_P)**2.0 + ejectile.mass**2.0) - ejectile.mass
print(f"Kinetic Energy (MeV): {ke:0.2f} Polar (deg): {np.rad2deg(result["polar"]):0.2f}")
ke = np.sqrt((estimate_gated["brho"][row]*float(ejectile.Z)*QBRHO_2_P)**2.0 + ejectile.mass**2.0) - ejectile.mass
print(f"Estimation KE: {ke:0.2f} Polar (deg): {np.rad2deg(estimate_gated["polar"][row]):0.2f}")

In [ ]:
plt.close("all")
fig, axs = plt.subplot_mosaic(
    """
    AB
    AC
    """,
    constrained_layout=True,
    figsize=(16,8)
)

rho_mm = guess.brho/det_params.magnetic_field * 1000.0 * np.sin(guess.polar)
length_samples = np.linspace(1.0, 50.0, 50)
dir_x_samples = length_samples * np.cos(guess.azimuthal) + guess.vertex_x
dir_y_samples = length_samples * np.sin(guess.azimuthal) + guess.vertex_y
circle_points = generate_circle_points(estimate_gated["center_x"][row], estimate_gated["vertex_y"][row], rho_mm)

axs["A"].scatter(cluster.data[:, 0]*0.001, cluster.data[:, 1]*0.001, s=3, label="Cluster")
axs["A"].plot(circle_points[:,0]*0.001,circle_points[:,1]*0.001, color='blue')
axs["A"].plot(best_fit_trajectory[:, 0], best_fit_trajectory[:, 1],label="Best Fit",zorder=1,color='red')
axs["A"].scatter([result["vertex_x"]], [result["vertex_y"]], s=3, label="Best Fit Vertex")
axs["A"].scatter
axs["A"].set_xlabel("X (m)")
axs["A"].set_ylabel("Y (m)")
axs["A"].set_xlim(-0.3, 0.3)
axs["A"].set_ylim(-0.3, 0.3)
axs["B"].scatter(cluster.data[:, 2]*0.001, cluster.data[:, 1]*0.001, s=3, label="Cluster")
axs["B"].plot(best_fit_trajectory[:, 2], best_fit_trajectory[:, 1],  label="Best Fit",zorder=1,color='red')
axs["B"].scatter([result["vertex_z"]], [result["vertex_y"]], s=3, label="Best Fit Vertex")
axs["B"].set_xlabel("Z (m)")
axs["B"].set_ylabel("Y (m)")
axs["B"].set_xlim(.0, 1.0)
axs["B"].set_ylim(-0.3, 0.3)
axs["C"].scatter(cluster.data[:, 2]*0.001, cluster.data[:, 0]*0.001, s=3, label="Cluster")
axs["C"].plot(best_fit_trajectory[:, 2], best_fit_trajectory[:, 0],  label="Best Fit",zorder=1,color='red')
axs["C"].scatter([result["vertex_z"]], [result["vertex_x"]], s=3, label="Best Fit Vertex")
axs["C"].set_xlabel("Z (m)")
axs["C"].set_ylabel("X (m)")
axs["C"].set_xlim(.0, 1.0)
axs["C"].set_ylim(-0.3, 0.3)
axs["A"].legend()
axs["A"].grid()
axs["B"].legend()
axs["B"].grid()
axs["C"].legend()
axs["C"].grid()
fig.savefig("delete.png")

In [ ]:
# Add specific conditions to trim the data. 
# along "z" it will always try to fit 


# Manually trim the cluster data

In [ ]:
fig, axs = plt.subplot_mosaic(
    """
    AB
    AC
    """,
    constrained_layout=True,
    figsize=(16,8)
)

# new cluster 
mask = (cluster.data[:,1]*0.001<0.05284) & (cluster.data[:,0]*0.001>-0.08)

axs["A"].scatter(cluster.data[:, 0]*0.001, cluster.data[:, 1]*0.001, s=3, color='black', label="Cluster")
axs["A"].scatter(cluster.data[mask, 0]*0.001, cluster.data[mask, 1]*0.001, s=3, color='blue', label="Cluster")


axs["A"].set_xlabel("X (m)")
axs["A"].set_ylabel("Y (m)")
axs["A"].set_xlim(-0.3, 0.3)
axs["A"].set_ylim(-0.3, 0.3)
axs["B"].scatter(cluster.data[:, 2]*0.001, cluster.data[:, 1]*0.001, s=3, color='black',label="Cluster")
axs["B"].scatter(cluster.data[mask, 2]*0.001, cluster.data[mask, 1]*0.001, s=3, color='blue',label="Cluster")
axs["B"].set_xlabel("Z (m)")
axs["B"].set_ylabel("Y (m)")
axs["B"].set_xlim(.0, 1.0)
axs["B"].set_ylim(-0.3, 0.3)
axs["C"].scatter(cluster.data[:, 2]*0.001, cluster.data[:, 0]*0.001, s=3, color='black',label="Cluster")
axs["C"].scatter(cluster.data[:, 2]*0.001, cluster.data[:, 0]*0.001, s=3, color='blue', label="Cluster")
axs["C"].set_xlabel("Z (m)")
axs["C"].set_ylabel("X (m)")
axs["C"].set_xlim(.0, 1.0)
axs["C"].set_ylim(-0.3, 0.3)
axs["A"].legend()
axs["A"].grid()
axs["B"].legend()
axs["B"].grid()
axs["C"].legend()
axs["C"].grid()
fig.savefig("delete.png")

In [ ]:
new_cluster = Cluster(cluster.event, cluster.label, cluster.direction, cluster.data[mask,:])

In [ ]:
result = fit_model_interp(new_cluster, guess, pid.nucleus, tracks, det_params, solver_params)
if result is None:
    print("Guess outside of interpolation range!")

best_fit_trajectory = interpolate_trajectory(result, tracks, pid.nucleus)

print(f"Vertex: ({result["vertex_x"].value*1000:0.2f}, {result["vertex_y"].value*1000:0.2f}, {result["vertex_z"].value*1000:0.2f})")
ejectile = nuclear_map.get_data(1,1)
ke = np.sqrt((result["brho"].value*float(ejectile.Z)*QBRHO_2_P)**2.0 + ejectile.mass**2.0) - ejectile.mass
print(f"Kinetic Energy (MeV): {ke:0.2f} Polar (deg): {np.rad2deg(result["polar"]):0.2f}")
ke = np.sqrt((estimate_gated["brho"][row]*float(ejectile.Z)*QBRHO_2_P)**2.0 + ejectile.mass**2.0) - ejectile.mass
print(f"Estimation KE: {ke:0.2f} Polar (deg): {np.rad2deg(estimate_gated["polar"][row]):0.2f}")

In [ ]:
fig, axs = plt.subplot_mosaic(
    """
    AB
    AC
    """,
    constrained_layout=True,
    figsize=(16,8)
)

axs["A"].scatter(new_cluster.data[:, 0]*0.001, new_cluster.data[:, 1]*0.001, s=3, label="Cluster")
axs["A"].plot(best_fit_trajectory[:, 0], best_fit_trajectory[:, 1],label="Best Fit",zorder=1,color='red')
axs["A"].scatter([result["vertex_x"]], [result["vertex_y"]], s=3, label="Best Fit Vertex")
axs["A"].set_xlabel("X (m)")
axs["A"].set_ylabel("Y (m)")
axs["A"].set_xlim(-0.3, 0.3)
axs["A"].set_ylim(-0.3, 0.3)
axs["B"].scatter(new_cluster.data[:, 2]*0.001, new_cluster.data[:, 1]*0.001, s=3, label="Cluster")
axs["B"].plot(best_fit_trajectory[:, 2], best_fit_trajectory[:, 1],  label="Best Fit",zorder=1,color='red')
axs["B"].scatter([result["vertex_z"]], [result["vertex_y"]], s=3, label="Best Fit Vertex")
axs["B"].set_xlabel("Z (m)")
axs["B"].set_ylabel("Y (m)")
axs["B"].set_xlim(.0, 1.0)
axs["B"].set_ylim(-0.3, 0.3)
axs["C"].scatter(new_cluster.data[:, 2]*0.001, new_cluster.data[:, 0]*0.001, s=3, label="Cluster")
axs["C"].plot(best_fit_trajectory[:, 2], best_fit_trajectory[:, 0],  label="Best Fit",zorder=1,color='red')
axs["C"].scatter([result["vertex_z"]], [result["vertex_x"]], s=3, label="Best Fit Vertex")
axs["C"].set_xlabel("Z (m)")
axs["C"].set_ylabel("X (m)")
axs["C"].set_xlim(.0, 1.0)
axs["C"].set_ylim(-0.3, 0.3)
axs["A"].legend()
axs["A"].grid()
axs["B"].legend()
axs["B"].grid()
axs["C"].legend()
axs["C"].grid()
fig.savefig("delete.png")